In [1]:
# import libraries

import numpy as np
import pandas as pd
import geopandas as gpd

import matplotlib.pyplot as plt

import os

Load files

In [2]:
# Load total pop
total_pop_folder = "Population Data"
pop_gdf = gpd.read_parquet(os.path.join(total_pop_folder, "swk_1km_2020_population_age_breakdown.parquet"))

# Load student pop
student_pop_gdf = gpd.read_parquet(os.path.join(total_pop_folder, "swk_1km_2020_population_stu_breakdown.parquet"))

# Load school pop
school_folder = "School Data"
school_gdf = gpd.read_parquet(os.path.join(school_folder, "swk_list_of_schools_2025.parquet"))

# Load combined routes
routes_folder = "OSRM Routes to Nearest School"
secondary_combined_routes_gdf = gpd.read_parquet(os.path.join(routes_folder, "secondary_combined_routes.parquet"))
primary_combined_routes_gdf = gpd.read_parquet(os.path.join(routes_folder, "primary_combined_routes.parquet"))

# Boundaries
boundary_folder = "Geographic Boundaries"
swk_districts_gdf = gpd.read_file(os.path.join(boundary_folder, "sarawak_districts.geojson")) # District boundaries
swk_parlimen_gdf = gpd.read_file(os.path.join(boundary_folder, "sarawak_parliament.geojson")) # Parlimen boundaries
swk_dun_gdf = gpd.read_file(os.path.join(boundary_folder, "sarawak_dun.geojson")) # DUN boundaries

In [3]:
# Create total, primary and secondary population gdfs
total_pop_gdf = pop_gdf.copy()[["id","total_pop","x","y","district","geometry"]]
primary_pop_gdf = student_pop_gdf.copy()[["id","primary_school_students","x","y","district","geometry"]]
secondary_pop_gdf = student_pop_gdf.copy()[["id","secondary_school_students","x","y","district","geometry"]]

# Round numbders
total_pop_gdf["total_pop"] = total_pop_gdf["total_pop"].round(0).astype(int)
primary_pop_gdf["primary_school_students"] = primary_pop_gdf["primary_school_students"].round(0).astype(int)
secondary_pop_gdf["secondary_school_students"] = secondary_pop_gdf["secondary_school_students"].round(0).astype(int)

# Rename columns
total_pop_gdf = total_pop_gdf.rename(columns={"id":"pop_id","total_pop":"total_population"})
primary_pop_gdf = primary_pop_gdf.rename(columns={"id":"pop_id"})
secondary_pop_gdf = secondary_pop_gdf.rename(columns={"id":"pop_id"})

In [4]:
# Clean routes data
def clean_combined_routes_gdf(combined_routes_gdf):
    # Remove columns
    combined_routes_gdf = combined_routes_gdf.drop(columns=["euclidean_km"])
    
    # Rename columns
    combined_routes_gdf = combined_routes_gdf.rename(columns={
        "combined_route": "travel_mode",
        "osrm_km": "dist_km",
        "osrm_min": "time_min"
    })
    
    # Round numbers
    combined_routes_gdf["dist_km"] = combined_routes_gdf["dist_km"].round(1)
    combined_routes_gdf["time_min"] = combined_routes_gdf["time_min"].round(0)
    return combined_routes_gdf

secondary_combined_routes_gdf = clean_combined_routes_gdf(secondary_combined_routes_gdf)
primary_combined_routes_gdf = clean_combined_routes_gdf(primary_combined_routes_gdf)

In [5]:
# Create primary and school gdf
primary_school_gdf = school_gdf.copy()[school_gdf["primary_secondary"]=="Primary"][["id","nama_sekolah","bil_murid","bil_guru","geometry"]]
secondary_school_gdf = school_gdf.copy()[school_gdf["primary_secondary"]=="Secondary"][["id","nama_sekolah","bil_murid","bil_guru","geometry"]]

# Rename columns
primary_school_gdf = primary_school_gdf.rename(columns={"id":"school_id","nama_sekolah":"school_name","bil_murid":"num_students","bil_guru":"num_teachers"})
secondary_school_gdf = secondary_school_gdf.rename(columns={"id":"school_id","nama_sekolah":"school_name","bil_murid":"num_students","bil_guru":"num_teachers"})

In [6]:
# merge population gdf with combined routes gdf
secondary_pop_routes_gdf = secondary_pop_gdf[["pop_id","secondary_school_students","district"]].merge(
    secondary_combined_routes_gdf,
    on="pop_id",
    how="left"
)
secondary_pop_routes_gdf = secondary_pop_routes_gdf.rename(columns={"secondary_school_students":"students"})

primary_pop_routes_gdf = primary_pop_gdf[["pop_id","primary_school_students","district"]].merge(
    primary_combined_routes_gdf,
    on="pop_id",
    how="left"
)
primary_pop_routes_gdf = primary_pop_routes_gdf.rename(columns={"primary_school_students":"students"})

In [7]:
# Clean boundaries data

# Districts
swk_districts_gdf = swk_districts_gdf[["name","geometry"]].rename(columns={"name":"district"})

# DUNs
swk_dun_gdf = swk_dun_gdf[["dun","geometry"]]
swk_parlimen_gdf = swk_parlimen_gdf[["parlimen","geometry"]]

Plot the distribution of travel time of 40 districts

In [8]:
# Optional: folder to save the plots
out_dir = "District Charts/Travel Time Distribution_Charts"
os.makedirs(out_dir, exist_ok=True)

# --- Common bins for comparability across districts ---
max_time = max(
    primary_pop_routes_gdf["time_min"].max(),
    secondary_pop_routes_gdf["time_min"].max()
)

bin_width = 10  # minutes
bins = np.arange(0, np.ceil(max_time / bin_width) * bin_width + bin_width,
                 bin_width)

districts = sorted(primary_pop_routes_gdf["district"].unique())

for district in districts:
    # Filter by district
    prim = primary_pop_routes_gdf[
        primary_pop_routes_gdf["district"] == district
    ]
    sec = secondary_pop_routes_gdf[
        secondary_pop_routes_gdf["district"] == district
    ]

    # Skip if no data
    if prim.empty and sec.empty:
        continue

    fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharex=True, sharey=True)
    ax_p, ax_s = axes

    # ---------------- Primary ----------------
    ax_p.hist(
        prim["time_min"],
        bins=bins,
        weights=prim["students"],
        edgecolor="black",
        alpha=0.8
    )
    ax_p.set_title(f"{district} – Primary")
    ax_p.set_ylabel("Number of students")
    ax_p.set_xlabel("Travel time to school (minutes)")

    # ---------------- Secondary ----------------
    ax_s.hist(
        sec["time_min"],
        bins=bins,
        weights=sec["students"],
        edgecolor="black",
        alpha=0.8
    )
    ax_s.set_title(f"{district} – Secondary")
    ax_s.set_xlabel("Travel time to school (minutes)")

    # Shared formatting
    for ax in axes:
        ax.grid(axis="y", linestyle=":", alpha=0.4)
        #ax.set_xlim(0, 300)

    plt.tight_layout()

    # Save one figure per district (optional)
    fname = district.replace(" ", "_") + "_time_distribution.png"
    plt.savefig(os.path.join(out_dir, fname), dpi=300, bbox_inches="tight")
    plt.close(fig)